In [1]:
import openeo, numpy as np, pandas as pd

In [2]:
# Setup backend connection (used by all downstream processing blocks)
CONNECTION = openeo.connect("openeofed.dataspace.copernicus.eu")

# List collections available on the openEO back-end
CONNECTION.list_collection_ids()

# Get detailed metadata of a certain collection
CONNECTION.describe_collection("SENTINEL2_L2A")

# authentication

CONNECTION.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeofed.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

## Sentinel-2 monthly cloud-masked composite with polygons

The next cells create a monthly Sentinel-2 RGB composite, mask cloudy pixels using the Sentinel-2 Scene Classification Layer, and draw the selected polygons on top. The polygon color is passed as a parameter.

In [ ]:
from pathlib import Path
import calendar
import sys

import matplotlib.pyplot as plt


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "project" / "domain" / "regions.py").exists():
            return path
    raise FileNotFoundError("Could not find project/domain/regions.py")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project.domain.regions import BAKOOL_POLYGON, BURHAKABA_POLYGON, GEDO_POLYGON


def normalize_polygon(polygon):
    points = [(float(lon), float(lat)) for lon, lat in polygon]
    if points[0] != points[-1]:
        points.append(points[0])
    return points


def buffered_extent_for_polygons(polygons, buffer_fraction=0.10):
    points = [point for polygon in polygons for point in normalize_polygon(polygon)]
    lons = [lon for lon, _ in points]
    lats = [lat for _, lat in points]

    west, east = min(lons), max(lons)
    south, north = min(lats), max(lats)
    lon_buffer = (east - west) * buffer_fraction
    lat_buffer = (north - south) * buffer_fraction

    return {
        "west": west - lon_buffer,
        "east": east + lon_buffer,
        "south": south - lat_buffer,
        "north": north + lat_buffer,
    }


In [ ]:
def month_temporal_extent(year, month):
    last_day = calendar.monthrange(year, month)[1]
    return [f"{year}-{month:02d}-01", f"{year}-{month:02d}-{last_day:02d}"]


def sentinel2_monthly_rgb_cube(
    connection,
    polygons,
    year,
    month,
    max_cloud_cover=80,
    buffer_fraction=0.10,
):
    extent = buffered_extent_for_polygons(polygons, buffer_fraction=buffer_fraction)
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=extent,
        temporal_extent=month_temporal_extent(year, month),
        bands=["B02", "B03", "B04", "SCL"],
        properties={"eo:cloud_cover": lambda cloud_cover: cloud_cover <= max_cloud_cover},
    )

    scl = cube.band("SCL")
    cloud_or_invalid = (scl == 0) | (scl == 1) | (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10)
    rgb = cube.filter_bands(["B04", "B03", "B02"]).mask(cloud_or_invalid)

    monthly_mean = rgb.reduce_dimension(dimension="t", reducer="mean")
    png_ready = monthly_mean.linear_scale_range(0, 3000, 0, 255)
    return png_ready, extent


def download_sentinel2_monthly_png(
    connection,
    polygons,
    year,
    month,
    output_dir=REPO_ROOT / "project" / "data" / "sentinel2",
    max_cloud_cover=80,
    buffer_fraction=0.10,
    overwrite=False,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    raw_png = output_dir / f"sentinel2_monthly_rgb_{year}_{month:02d}.png"

    cube, extent = sentinel2_monthly_rgb_cube(
        connection=connection,
        polygons=polygons,
        year=year,
        month=month,
        max_cloud_cover=max_cloud_cover,
        buffer_fraction=buffer_fraction,
    )
    if overwrite or not raw_png.exists():
        cube.save_result(format="PNG").download(str(raw_png))

    return raw_png, extent


In [ ]:
def draw_polygons_on_sentinel2_png(
    image_path,
    extent,
    polygons,
    polygon_color="yellow",
    output_path=None,
    linewidth=2.5,
    figsize=(10, 10),
):
    image_path = Path(image_path)
    if output_path is None:
        output_path = image_path.with_name(f"{image_path.stem}_polygons.png")
    output_path = Path(output_path)

    image = plt.imread(image_path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(
        image,
        extent=[extent["west"], extent["east"], extent["south"], extent["north"]],
        origin="upper",
    )

    for polygon in polygons:
        points = normalize_polygon(polygon)
        xs = [lon for lon, _ in points]
        ys = [lat for _, lat in points]
        ax.plot(xs, ys, color=polygon_color, linewidth=linewidth)

    ax.set_xlim(extent["west"], extent["east"])
    ax.set_ylim(extent["south"], extent["north"])
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"Sentinel-2 monthly RGB composite: {image_path.stem}")
    ax.grid(color="white", alpha=0.25, linewidth=0.7)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.show()
    return output_path


In [ ]:
# Parameters
POLYGONS = [GEDO_POLYGON, BURHAKABA_POLYGON, BAKOOL_POLYGON]
YEAR = 2024
MONTH = 6
POLYGON_COLOR = "#ffcc00"
MAX_CLOUD_COVER = 80

raw_png, sentinel_extent = download_sentinel2_monthly_png(
    connection=CONNECTION,
    polygons=POLYGONS,
    year=YEAR,
    month=MONTH,
    max_cloud_cover=MAX_CLOUD_COVER,
)

overlay_png = draw_polygons_on_sentinel2_png(
    image_path=raw_png,
    extent=sentinel_extent,
    polygons=POLYGONS,
    polygon_color=POLYGON_COLOR,
)

overlay_png


box

1°01'26.38"N 46°10'54.35"E 
6°36'37.73"N 39°58'00.82"E